In [ ]:
from pathlib import Path

zip_path = Path("/content/Team6_training_handoff.zip")

print("ZIP exists:", zip_path.exists())

if zip_path.exists():
    print("ZIP size:", round(zip_path.stat().st_size / 1048576, 2), "MB")

ZIP exists: True
ZIP size: 26.5 MB


In [ ]:
import zipfile

extract_path = Path("/content/pathsnap_project")

extract_path.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    print("Files inside ZIP:", len(z.namelist()))
    z.extractall(extract_path)

print("Extraction completed!")


Files inside ZIP: 114
Extraction completed!


In [ ]:
import os

matches = list(
    extract_path.rglob("PRE_PROCESSING_DECISIONS.md")
)

assert len(matches) == 1, (
    f"Expected one project root, found {len(matches)}"
)

ROOT = matches[0].parent

os.chdir(ROOT)

print("Project directory:", ROOT)
print("\nProject files:")

for item in sorted(ROOT.iterdir()):
    print("📁" if item.is_dir() else "📄", item.name)

Project directory: /content/pathsnap_project

Project files:
📄 .gitignore
📄 PRE_PROCESSING_DECISIONS.md
📄 README.md
📁 docs
📁 processed
📁 reports
📄 requirements-notebooks.txt
📄 requirements-preprocessing.txt
📄 requirements-training.txt
📁 scripts
📁 tests


In [ ]:
import subprocess
import sys

for filename in [
    "requirements-notebooks.txt",
    "requirements-training.txt"
]:
    requirements = ROOT / filename

    if requirements.exists():
        print("Installing:", filename)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-r",
                str(requirements)
            ],
            check=True
        )

Installing: requirements-notebooks.txt
Installing: requirements-training.txt


In [ ]:
import sklearn
import pandas as pd
import numpy as np

print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Scikit-learn: 1.6.1
Pandas: 2.2.3
NumPy: 2.4.6


In [ ]:
files_to_read = [
    "PRE_PROCESSING_DECISIONS.md",
    "docs/TRAINING_GUIDE.md",
    "docs/TEAM_TASKS.md",
]

for name in files_to_read:
    path = ROOT / name

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    if path.exists():
        print(path.read_text(encoding="utf-8"))
    else:
        print("FILE NOT FOUND")


PRE_PROCESSING_DECISIONS.md
# PathSnap preprocessing decisions

## Purpose and status
One shared candidate dataset supports Logistic Regression, Histogram Gradient Boosting,
Random Forest, Extra Trees, and MLP. No model has been trained. Labels derived below
are **proxy labels**, not organiser-supplied observation-level ground truth.
Raw files are preserved. Only train and student_test are read; evaluator data is excluded.

## Representation
Each trip produces one compressed NumPy file. `X` has shape (GPS observations, 8 candidates,
10 features); `y` is binary candidate correctness. Train a binary candidate classifier on
flattened rows, then reshape positive-class scores and choose the best candidate per GPS
observation. Candidate IDs map scores back to actual road segments and remain metadata.
Arc IDs are local to their trip. Network file hashes differ across all trips in each partition.
Never use trip IDs, raw row IDs, reference IDs, or label diagnostics as predictive features.

## 

In [ ]:
import ast

for name in [
    "scripts/load_dataset.py",
    "scripts/model_training.py"
]:
    path = ROOT / name

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    for node in tree.body:
        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)
        ):
            print(f"Line {node.lineno}: {node.name}")

    print("\nSOURCE CODE:\n")
    print(source)


scripts/load_dataset.py
Line 5: load_training

SOURCE CODE:

"""Shared loader: IDs and derived-label diagnostics never enter X."""
from pathlib import Path
import numpy as np

def load_training(root='processed', validation_fold=None):
    """Return fit and validation dictionaries; validation retains ALL observations.

    X has shape (observations, candidates, features). Flatten candidate rows for
    binary classifier fitting; reshape positive probabilities for segment choice.
    Masks restrict fitting, never silently remove evaluation failures.
    """
    def collect(files, fitting):
        rows=[]
        for p in files:
            with np.load(p,allow_pickle=False) as d:
                mask=d['training_observation_mask'] if fitting else np.ones(len(d['X']),dtype=bool)
                rows.append((d['X'][mask],d['y'][mask],d['candidate_arc_ids'][mask],d['reference_arc_id'][mask],np.full(mask.sum(),p.stem),d['label_reliable'][mask]))
        if not rows:raise ValueError('No tri

In [ ]:
from sklearn.ensemble import RandomForestClassifier

SEED = 42

rf_params = {
    "n_estimators": 200,
    "criterion": "gini",
    "max_features": "sqrt",
    "min_samples_leaf": 2,
    "random_state": SEED,
    "n_jobs": -1
}

rf = RandomForestClassifier(**rf_params)

print("Initial Random Forest configuration:")
print(rf.get_params())


Initial Random Forest configuration:
{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 2, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 200, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

print("Saved files:\n")

for f in sorted(Path(SAVE_DIR).rglob("*")):
    if f.is_file():
        print(
            f.relative_to(SAVE_DIR),
            round(f.stat().st_size / 1024, 2),
            "KB"
        )

print("\nTraining exit code:")
print(process.returncode)

Saved files:

random_forest/configuration.json 0.44 KB
random_forest/fold_0_predictions.npz 924.02 KB
random_forest/metrics.json 2.13 KB
random_forest/provenance.json 9.5 KB

Training exit code:
None


In [ ]:
import json

metrics_path = Path(SAVE_DIR) / "random_forest" / "metrics.json"

with open(metrics_path) as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))

[
  {
    "split": "fold_0",
    "fit_seconds": 519.909308024,
    "inference_seconds": 3.086311225999907,
    "inference_ms_per_observation": 0.1075369765156762,
    "metrics": {
      "all_points": {
        "observations": 28700,
        "top1_accuracy": 0.7728919860627178,
        "macro_precision": 0.6443965157440722,
        "macro_recall": 0.5881429997421448,
        "macro_f1": 0.6050363832372773,
        "candidate_miss_rate": 0.00975609756097561,
        "categorical_log_loss_with_miss_state": 1.1812451296358315
      },
      "reliable_points": {
        "observations": 20087,
        "top1_accuracy": 0.8594115597152387,
        "macro_precision": 0.7644910999543587,
        "macro_recall": 0.7195935212123589,
        "macro_f1": 0.7358597951363389,
        "candidate_miss_rate": 0.0010952357245979987,
        "categorical_log_loss_with_miss_state": 0.409096881623782
      },
      "reliable_coverage": 0.6998954703832753,
      "candidate_binary_log_loss": 0.1782250413487404

In [ ]:
import json
import pandas as pd

# Let's inspect the metrics and summaries produced by the run
summary_file = SAVE_DIR / "summary.json"
metrics_file = SAVE_DIR / "metrics.json"

if summary_file.exists():
    print("--- SUMMARY ---")
    print(summary_file.read_text())

if metrics_file.exists():
    print("\n--- DETAILED METRICS ---")
    metrics_data = json.loads(metrics_file.read_text())
    df_metrics = pd.json_normalize(metrics_data)
    display(df_metrics)


In [ ]:
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the trained model
model_path = SAVE_DIR / "model.joblib"
if model_path.exists():
    pipeline = joblib.load(model_path)
    print("Loaded model:", pipeline)

    # Extract classifier and feature importance
    # In the pipeline, the last step is usually the classifier
    clf = pipeline.steps[-1][1]

    # Retrieve feature names if StandardScaler/pipeline supports or use default index
    # Let's inspect feature importances
    importances = clf.feature_importances_
    indices = np.argsort(importances)[::-1]

    # Assuming 10 standard features
    feature_names = [f"Feature {i}" for i in range(len(importances))]

    plt.figure(figsize=(10, 6))
    sns.barplot(x=importances[indices], y=[feature_names[i] for i in indices], palette="viridis")
    plt.title("Random Forest Feature Importances")
    plt.xlabel("Relative Importance")
    plt.ylabel("Features")
    plt.tight_layout()
    plt.savefig(SAVE_DIR / "feature_importances.png")
    plt.show()
else:
    print("Model file not found.")


Model file not found.


### Evaluation Summary & Findings

- **Overfitting & Generalisation:** Compare the training subset metrics vs validation/evaluation metrics in the displayed DataFrame. A typical Random Forest with deep trees may show extremely high training performance, but we must verify if the cross-validation score holds steady around the mean or if there is generalisation drop.
- **Limitations:** The predictions rely on proxy labels rather than observation-level ground truth. Furthermore, candidate misses remain in evaluation and should be accounted for when reviewing final pipeline accuracy.

In [ ]:
import numpy as np
from pathlib import Path

file_path = Path(SAVE_DIR) / "random_forest" / "fold_0_predictions.npz"

data = np.load(file_path, allow_pickle=False)

print("Arrays inside NPZ:")
for key in data.files:
    print(key, data[key].shape, data[key].dtype)

Arrays inside NPZ:
positive_probability (28700, 8) float64
predicted_arc_id (28700,) int64
reference_arc_id (28700,) int64
candidate_arc_ids (28700, 8) int64
trip_id (28700,) <U8
label_reliable (28700,) bool


In [ ]:
for key in data.files:
    print(f"\n--- {key} ---")
    print(data[key][:10])


--- positive_probability ---
[[6.06542535e-01 8.57275132e-02 0.00000000e+00 0.00000000e+00
  3.33333333e-03 0.00000000e+00 0.00000000e+00 8.46050870e-04]
 [9.95911760e-01 3.33333333e-03 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [9.98493597e-01 6.66666667e-03 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [9.97143401e-01 6.66666667e-03 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [9.89578880e-01 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [9.89020377e-01 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [8.95227599e-01 1.22688423e-02 0.00000000e+00 1.33333333e-02
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [9.44668258e-01 0.00000000e+00 3.73692078e-05 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0

In [ ]:
import pandas as pd

df = pd.DataFrame({
    key: data[key]
    for key in data.files
    if data[key].ndim == 1
})

display(df.head(20))

,predicted_arc_id,reference_arc_id,trip_id,label_reliable
0,64973,64973,00000001,True
1,64973,64973,00000001,True
2,64973,64973,00000001,True
3,64973,64973,00000001,True
4,64973,64973,00000001,True
5,64973,64973,00000001,True
6,133948,133948,00000001,True
7,133949,133949,00000001,True
8,139208,139208,00000001,True
9,139208,139208,00000001,False


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        str(ROOT / "scripts/model_training.py"),
        "--help"
    ],
    capture_output=True,
    text=True,
    cwd=str(ROOT)
)

print(result.stdout)
print(result.stderr)

usage: model_training.py [-h]
                         --model {logistic,hist_gradient_boosting,random_forest,extra_trees,mlp}
                         [--root ROOT] [--out OUT]
                         [--folds {0,1,2,3,4} [{0,1,2,3,4} ...]] [--final]
                         [--smoke]

options:
  -h, --help            show this help message and exit
  --model {logistic,hist_gradient_boosting,random_forest,extra_trees,mlp}
  --root ROOT
  --out OUT
  --folds {0,1,2,3,4} [{0,1,2,3,4} ...]
  --final               Also refit all training trips and evaluate student-
                        test
  --smoke               First 200 eligible observations per split; diagnostic
                        only




In [ ]:
import subprocess
import sys
import time
from pathlib import Path

# Save separately to protect your completed Fold 0
NEW_OUT = Path(SAVE_DIR) / "remaining_run"
NEW_OUT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "-u",
    str(ROOT / "scripts/model_training.py"),
    "--model", "random_forest",
    "--root", str(ROOT / "processed"),
    "--out", str(NEW_OUT),
    "--folds", "1", "2", "3", "4",
    "--final"
]

print("Starting Folds 1–4 + final student-testing")
print("Output directory:", NEW_OUT)

start = time.time()

process = subprocess.Popen(
    cmd,
    cwd=str(ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="", flush=True)

process.wait()

print("\nExit code:", process.returncode)
print("Elapsed minutes:", round((time.time() - start) / 60, 2))

Starting Folds 1–4 + final student-testing
Output directory: /content/drive/MyDrive/PathSnap_RandomForest/remaining_run
fold_1 {"split": "fold_1", "fit_seconds": 465.54486473399993, "inference_seconds": 5.342793952000193, "inference_ms_per_observation": 0.13129193374945183, "metrics": {"all_points": {"observations": 40694, "top1_accuracy": 0.753477171081732, "macro_precision": 0.7214135975467728, "macro_recall": 0.6767167101353083, "macro_f1": 0.6904985821915496, "candidate_miss_rate": 0.10217722514375584, "categorical_log_loss_with_miss_state": 3.338810727935243}, "reliable_points": {"observations": 29107, "top1_accuracy": 0.8747380355241007, "macro_precision": 0.8376507724301843, "macro_recall": 0.8063219071616291, "macro_f1": 0.8177185633749735, "candidate_miss_rate": 0.017177998419624144, "categorical_log_loss_with_miss_state": 0.7732816395927217}, "reliable_coverage": 0.7152651496535116, "candidate_binary_log_loss": 0.12818948154966076, "route_continuity": null, "route_continuity_

In [22]:
import json
from pathlib import Path

print("SAVED FILES:")
for f in sorted(NEW_OUT.rglob("*")):
    if f.is_file():
        print(f.relative_to(NEW_OUT))

print("\nMETRICS:")
for f in NEW_OUT.rglob("metrics.json"):
    results = json.loads(f.read_text())

    for result in results:
        m = result.get("metrics", {}).get("all_points", {})
        print(
            result.get("split"),
            "Accuracy:", m.get("top1_accuracy"),
            "Macro F1:", m.get("macro_f1")
        )

SAVED FILES:
random_forest/configuration.json
random_forest/fold_1_predictions.npz
random_forest/fold_2_predictions.npz
random_forest/fold_3_predictions.npz
random_forest/fold_4_predictions.npz
random_forest/metrics.json
random_forest/model.joblib
random_forest/provenance.json
random_forest/student_test_predictions.npz
random_forest/summary.json

METRICS:
fold_1 Accuracy: 0.753477171081732 Macro F1: 0.6904985821915496
fold_2 Accuracy: 0.4677491555772596 Macro F1: 0.5898326473654921
fold_3 Accuracy: 0.7471836184485028 Macro F1: 0.6475919597126198
fold_4 Accuracy: 0.8526648042665893 Macro F1: 0.7076029824013633
student_test Accuracy: 0.8096555323590814 Macro F1: 0.6139620437933371


In [23]:
import joblib
import json
from pathlib import Path

original = Path(SAVE_DIR) / "random_forest"
remaining = Path(NEW_OUT) / "random_forest"

# Verify all prediction files
assert (original / "fold_0_predictions.npz").exists()

for i in range(1, 5):
    assert (remaining / f"fold_{i}_predictions.npz").exists()

assert (remaining / "student_test_predictions.npz").exists()

# Verify saved model loads
model = joblib.load(remaining / "model.joblib")

print("Model loaded:", type(model).__name__)
print("Trees:", getattr(model, "n_estimators", "N/A"))

# Verify recorded splits
records = (
    json.loads((original / "metrics.json").read_text())
    + json.loads((remaining / "metrics.json").read_text())
)

print("Recorded splits:", [r["split"] for r in records])
print("All predictions and model verified!")

Model loaded: RandomForestClassifier
Trees: 150
Recorded splits: ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4', 'student_test']
All predictions and model verified!


In [24]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# Locate both training runs
FOLD0 = Path(SAVE_DIR) / "random_forest"
REMAINING = Path(NEW_OUT) / "random_forest"

def read_json(path):
    return json.loads(path.read_text()) if path.exists() else None

# Combine metrics
records = []
for folder in [FOLD0, REMAINING]:
    data = read_json(folder / "metrics.json")
    if isinstance(data, list):
        records.extend(data)

records = sorted(
    records,
    key=lambda r: (
        1 if r.get("split") == "student_test" else 0,
        r.get("split", "")
    )
)

print("=" * 75)
print("TEAM 6 — PATHSNAP RANDOM FOREST COMPLETE EXPERIMENT SUMMARY")
print("=" * 75)

# 1. Main metrics
rows = []
for r in records:
    m = r.get("metrics", {})
    a = m.get("all_points", {})
    reliable = m.get("reliable_points", {})
    train = r.get("fit_subset_metrics", {}).get("all_points", {})

    rows.append({
        "Split": r.get("split"),
        "Observations": a.get("observations"),
        "Top-1 (%)": 100*a["top1_accuracy"] if a.get("top1_accuracy") is not None else None,
        "Macro F1 (%)": 100*a["macro_f1"] if a.get("macro_f1") is not None else None,
        "Precision (%)": 100*a["macro_precision"] if a.get("macro_precision") is not None else None,
        "Recall (%)": 100*a["macro_recall"] if a.get("macro_recall") is not None else None,
        "Miss Rate (%)": 100*a["candidate_miss_rate"] if a.get("candidate_miss_rate") is not None else None,
        "Categorical Log Loss": a.get("categorical_log_loss_with_miss_state"),
        "Binary Log Loss": m.get("candidate_binary_log_loss"),
        "Reliable Top-1 (%)": 100*reliable["top1_accuracy"] if reliable.get("top1_accuracy") is not None else None,
        "Reliable Coverage (%)": 100*m["reliable_coverage"] if m.get("reliable_coverage") is not None else None,
        "Fit Top-1 (%)": 100*train["top1_accuracy"] if train.get("top1_accuracy") is not None else None,
        "Fit Seconds": r.get("fit_seconds"),
        "Inference Seconds": r.get("inference_seconds"),
        "Inference ms/Obs": r.get("inference_ms_per_observation")
    })

df = pd.DataFrame(rows)

print("\n1. COMPLETE FOLD AND STUDENT-TEST METRICS")
display(df.round(4))

# 2. CV averages
cv = df[df["Split"].str.startswith("fold_")]

print("\n2. FIVE-FOLD CROSS-VALIDATION SUMMARY")
if len(cv) == 5:
    for col in ["Top-1 (%)", "Macro F1 (%)",
                "Precision (%)", "Recall (%)",
                "Miss Rate (%)"]:
        print(
            f"{col}: Mean = {cv[col].mean():.4f}, "
            f"Std = {cv[col].std():.4f}"
        )
else:
    print(f"WARNING: Found {len(cv)} of 5 folds")

# 3. Generalisation
print("\n3. TRAINING VS VALIDATION")
for _, r in cv.iterrows():
    if pd.notna(r["Fit Top-1 (%)"]):
        print(
            f"{r['Split']}: "
            f"Training={r['Fit Top-1 (%)']:.2f}% | "
            f"Validation={r['Top-1 (%)']:.2f}% | "
            f"Gap={r['Fit Top-1 (%)']-r['Top-1 (%)']:.2f} pp"
        )

# 4. Execution timing
print("\n4. COMPUTATIONAL PERFORMANCE")
print("Recorded fit time (minutes):",
      round(df["Fit Seconds"].sum()/60, 2))
print("Recorded inference time (seconds):",
      round(df["Inference Seconds"].sum(), 2))

# 5. Model configuration
print("\n5. MODEL CONFIGURATION")
config = read_json(REMAINING / "configuration.json")
print(json.dumps(config, indent=2, default=str))

# 6. Final model
print("\n6. SAVED FINAL MODEL")
model_path = REMAINING / "model.joblib"

if model_path.exists():
    model = joblib.load(model_path)
    print("Model type:", type(model).__name__)
    print("Model path:", model_path)
    print("Parameters:")
    if hasattr(model, "get_params"):
        print(json.dumps(model.get_params(), indent=2, default=str))
else:
    print("Final model NOT FOUND")

# 7. Feature importance
print("\n7. FEATURE IMPORTANCE")
if model_path.exists():
    estimator = model.steps[-1][1] if hasattr(model, "steps") else model

    if hasattr(estimator, "feature_importances_"):
        importances = estimator.feature_importances_

        names = getattr(model, "feature_names_in_", None)
        if names is None:
            names = getattr(estimator, "feature_names_in_", None)

        if names is not None and len(names) == len(importances):
            importance_df = pd.DataFrame({
                "Feature": names,
                "Importance": importances
            }).sort_values("Importance", ascending=False)
            display(importance_df)
        else:
            print(
                "Feature names unavailable in saved model.",
                "Importance values by index:"
            )
            print(importances)
    else:
        print("Feature importance not exposed by saved model")

# 8. Warnings
print("\n8. WARNINGS AND LIMITATIONS")
for r in records:
    print(r["split"], "Warnings:", r.get("warnings", []))
    print("Route continuity:",
          r.get("metrics", {}).get("route_continuity"))
    reason = r.get("metrics", {}).get("route_continuity_reason")
    if reason:
        print("Reason:", reason)

# 9. Saved artifacts
print("\n9. SAVED OUTPUT FILES")
for folder in [FOLD0, REMAINING]:
    print("\nDirectory:", folder)
    if folder.exists():
        for f in sorted(folder.rglob("*")):
            if f.is_file():
                print(f.name, f"({f.stat().st_size/1024:.2f} KB)")

# 10. Final summary
print("\n10. FINAL RUN SUMMARY")
summary = read_json(REMAINING / "summary.json")
print(json.dumps(summary, indent=2, default=str))

print("\n" + "=" * 75)
print("END OF RANDOM FOREST EXPERIMENT SUMMARY")
print("=" * 75)

TEAM 6 — PATHSNAP RANDOM FOREST COMPLETE EXPERIMENT SUMMARY

1. COMPLETE FOLD AND STUDENT-TEST METRICS


,Split,Observations,Top-1 (%),Macro F1 (%),Precision (%),Recall (%),Miss Rate (%),Categorical Log Loss,Binary Log Loss,Reliable Top-1 (%),Reliable Coverage (%),Fit Top-1 (%),Fit Seconds,Inference Seconds,Inference ms/Obs
0,fold_0,28700,77.2892,60.5036,64.4397,58.8143,0.9756,1.1812,0.1782,85.9412,69.9895,94.6617,519.9093,3.0863,0.1075
1,fold_1,40694,75.3477,69.0499,72.1414,67.6717,10.2177,3.3388,0.1282,87.4738,71.5265,94.2561,465.5449,5.3428,0.1313
2,fold_2,49146,46.7749,58.9833,63.1242,57.0400,28.5496,10.3955,0.4199,81.3681,45.8674,94.7967,502.1557,4.1799,0.0851
3,fold_3,62314,74.7184,64.7592,68.8062,62.7465,0.5633,0.8614,0.1577,77.4836,75.0345,97.2150,382.1984,6.8221,0.1095
4,fold_4,27563,85.2665,70.7603,73.7988,69.5623,0.5333,0.6974,0.1220,91.5618,67.7611,94.5342,527.6578,4.3551,0.1580
5,student_test,19160,80.9656,61.3962,65.6351,59.7707,0.3497,0.7780,0.1415,89.0283,69.4520,94.3550,622.1549,2.4317,0.1269



2. FIVE-FOLD CROSS-VALIDATION SUMMARY
Top-1 (%): Mean = 71.8793, Std = 14.6527
Macro F1 (%): Mean = 64.8113, Std = 5.1449
Precision (%): Mean = 68.4620, Std = 4.6586
Recall (%): Mean = 63.1670, Std = 5.4280
Miss Rate (%): Mean = 8.1679, Std = 12.1188

3. TRAINING VS VALIDATION
fold_0: Training=94.66% | Validation=77.29% | Gap=17.37 pp
fold_1: Training=94.26% | Validation=75.35% | Gap=18.91 pp
fold_2: Training=94.80% | Validation=46.77% | Gap=48.02 pp
fold_3: Training=97.22% | Validation=74.72% | Gap=22.50 pp
fold_4: Training=94.53% | Validation=85.27% | Gap=9.27 pp

4. COMPUTATIONAL PERFORMANCE
Recorded fit time (minutes): 50.33
Recorded inference time (seconds): 26.22

5. MODEL CONFIGURATION
{
  "bootstrap": true,
  "ccp_alpha": 0.0,
  "class_weight": null,
  "criterion": "gini",
  "max_depth": 24,
  "max_features": "sqrt",
  "max_leaf_nodes": null,
  "max_samples": null,
  "min_impurity_decrease": 0.0,
  "min_samples_leaf": 2,
  "min_samples_split": 2,
  "min_weight_fraction_leaf": 

In [25]:
import os
import json

folder = "/content/drive/MyDrive/PathSnap_RandomForest/remaining_run/random_forest"

json_files = [
    "configuration.json",
    "metrics.json",
    "provenance.json",
    "summary.json"
]

for filename in json_files:
    filepath = os.path.join(folder, filename)

    print("\n" + "="*70)
    print(f"FILE: {filename}")
    print("="*70)

    with open(filepath, "r") as f:
        data = json.load(f)

    print(json.dumps(data, indent=4))


FILE: configuration.json
{
    "bootstrap": true,
    "ccp_alpha": 0.0,
    "class_weight": null,
    "criterion": "gini",
    "max_depth": 24,
    "max_features": "sqrt",
    "max_leaf_nodes": null,
    "max_samples": null,
    "min_impurity_decrease": 0.0,
    "min_samples_leaf": 2,
    "min_samples_split": 2,
    "min_weight_fraction_leaf": 0.0,
    "monotonic_cst": null,
    "n_estimators": 150,
    "n_jobs": 2,
    "oob_score": false,
    "random_state": 212,
    "verbose": 0,
    "warm_start": false
}

FILE: metrics.json
[
    {
        "split": "fold_1",
        "fit_seconds": 465.54486473399993,
        "inference_seconds": 5.342793952000193,
        "inference_ms_per_observation": 0.13129193374945183,
        "metrics": {
            "all_points": {
                "observations": 40694,
                "top1_accuracy": 0.753477171081732,
                "macro_precision": 0.7214135975467728,
                "macro_recall": 0.6767167101353083,
                "macro_f1": 0.69